# SkinAI — YOLOv8 5-Class Skin Issue Training

**Dataset:** Skin Issues v6 (Roboflow) — Acne, Black-heads, Large Pores, Pigmentation, Rosacea

**Steps:**
1. Install dependencies
2. Upload dataset (or mount Google Drive)
3. Train YOLOv8n
4. Evaluate
5. Download best.pt

## 1. Install Dependencies

In [ ]:
!pip install ultralytics roboflow
!yolo checks

## 2. Upload Dataset

Option A: Upload the `Skin Issues.v6i.yolov8` folder to Colab
Option B: Mount Google Drive if dataset is stored there

In [ ]:
# Option A: Upload via Colab file browser
# Drag & drop the dataset folder to /content/

# Option B: Mount Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# DATASET_PATH = '/content/drive/MyDrive/Skin Issues.v6i.yolov8'

import os
DATASET_PATH = 'Skin Issues.v6i.yolov8'  # adjust if needed

data_yaml = os.path.join(DATASET_PATH, 'data.yaml')
assert os.path.exists(data_yaml), f'Data YAML not found: {data_yaml}'

# Fix relative paths in data.yaml to absolute paths
import yaml
with open(data_yaml, 'r') as f:
    cfg = yaml.safe_load(f)

cfg['train'] = os.path.join(DATASET_PATH, 'train', 'images')
cfg['val'] = os.path.join(DATASET_PATH, 'valid', 'images')
cfg['test'] = os.path.join(DATASET_PATH, 'test', 'images')

with open(data_yaml, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

print(f'Dataset: {DATASET_PATH}')
print(f'Classes: {cfg["names"]}')
print(f'Train: {len(os.listdir(cfg["train"]))} images')
print(f'Val:   {len(os.listdir(cfg["val"]))} images')
print(f'Test:  {len(os.listdir(cfg["test"]))} images')
print(f'\nData YAML updated with absolute paths:')
print(open(data_yaml).read())

## 3. Train YOLOv8n

In [ ]:
from ultralytics import YOLO

# Load pretrained YOLOv8 nano
model = YOLO('yolov8n.pt')

# Train
results = model.train(
    data=data_yaml,
    epochs=100,
    imgsz=640,
    batch=16,
    name='skinai_v2_5class',
    project='runs/detect',
    patience=20,          # early stopping
    # Augmentation
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=15.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.2,
    copy_paste=0.1,
    # Config
    workers=8,
    device=0,
    exist_ok=True,
    pretrained=True,
    optimizer='auto',
    verbose=True,
    seed=42,
    val=True,
    plots=True,
)

print('\n✅ Training complete!')

## 4. Evaluate

In [ ]:
# Load best model and evaluate
best_model = YOLO('runs/detect/skinai_v2_5class/weights/best.pt')
val_results = best_model.val(data=data_yaml, imgsz=640, batch=16)

print(f'\n{"="*50}')
print(f'mAP50:      {val_results.box.map50:.4f}')
print(f'mAP50-95:   {val_results.box.map:.4f}')
print(f'Precision:  {val_results.box.mp:.4f}')
print(f'Recall:     {val_results.box.mr:.4f}')

print(f'\n{"="*50}')
print('Per-class metrics:')
print(f'{"Class":15s} {"P":>8s} {"R":>8s} {"mAP50":>8s} {"mAP50-95":>8s}')
print('-' * 50)
names = val_results.names
for i, (p, r, ap50, ap) in enumerate(
    zip(val_results.box.p, val_results.box.r, val_results.box.ap50, val_results.box.ap)
):
    print(f'{names[i]:15s} {p:8.3f} {r:8.3f} {ap50:8.3f} {ap:8.3f}')

## 5. Test on Sample Images

In [ ]:
import glob
from IPython.display import display, Image

# Run inference on test images
test_images = glob.glob(os.path.join(DATASET_PATH, 'test', 'images', '*.jpg'))[:5]

for img_path in test_images:
    results = best_model.predict(img_path, conf=0.25, verbose=False)
    annotated = results[0].plot()
    print(f'\n{os.path.basename(img_path)}:')
    for box in results[0].boxes:
        cls = results[0].names[int(box.cls[0])]
        conf = float(box.conf[0])
        print(f'  {cls}: {conf:.2f}')
    display(Image(filename=results[0].save_dir + '/predict/' + os.path.basename(img_path)))

## 6. Export & Download

Download `best.pt` and place it in `backend/models/best.pt`

In [ ]:
import shutil

best_path = 'runs/detect/skinai_v2_5class/weights/best.pt'
assert os.path.exists(best_path), 'best.pt not found'

# Copy to working dir for easy download
shutil.copy(best_path, 'best.pt')
print(f'\nModel saved to: {os.path.abspath("best.pt")}')
print(f'Model size: {os.path.getsize("best.pt") / 1024 / 1024:.1f} MB')

# Download from Colab
from google.colab import files
files.download('best.pt')

print('\n✅ Download best.pt and place it in backend/models/best.pt')

## 7. Confusion Matrix & Curves

In [ ]:
from IPython.display import Image, display

plot_dir = 'runs/detect/skinai_v2_5class'
for plot_file in ['confusion_matrix.png', 'results.png', 'F1_curve.png', 'PR_curve.png']:
    path = os.path.join(plot_dir, plot_file)
    if os.path.exists(path):
        print(f'\n{plot_file}:')
        display(Image(filename=path, width=800))